In [1]:
# numpy는 숫자 여러 개를 배열로 묶어 한 번에 계산하게 해 주는 라이브러리
# 이 노트북에서는 데이터 준비/정규화/학습 전 확인 같은 'NumPy 흐름'에 사용
import numpy as np

# torch는 PyTorch 라이브러리
# 이번 노트북에서는 (1) 자동 미분(autograd)과 (2) optimizer(torch.optim.SGD)를 쓰기 위해 사용
# - 자동 미분 : 사람이 미분 공식을 적지 않아도 PyTorch가 a.grad, b.grad를 계산해 주는 기능
# - optimizer: 사람이 직접 하던 a, b 업데이트를 PyTorch가 대신 수행해 주는 도구
import torch

In [2]:
# 1. 입력값 X와 정답 y 준비(이전 파일과 동일)

# X: 입력값으로, 여기서는 사람의 키(cm)를 사용
#    np.array([...])는 여러 숫자를 하나의 NumPy 배열로 묶는 것
X = np.array([160, 170, 180, 190])

# y: 정답값으로, 0은 농구선수 아님, 1은 농구선수임
#    x와 y는 순서대로 짝지어짐. 즉, 키 160cm -> 정답 0, 키 190cm -> 정답 1
y = np.array([0, 0, 1, 1])

print('입력값 X:', X)
print('정답값 y:', y)

입력값 X: [160 170 180 190]
정답값 y: [0 0 1 1]


In [3]:
# 2. 입력값 정규화 (이전 파일과 동일)

# 평균과 표준편차를 계산
# - 평균: 데이터의 중심(가운데쯤 되는 값)
# - 표준편차: 데이터가 평균에서 얼마나 넓게 퍼져 있는지를 나타내는 값
X_mean = np.mean(X)
X_std = np.std(X)

# 정규화 공식: (원본값 - 평균) / 표준편차
# 입력값의 범위를 0 근처로 비슷하게 맞추면 학습이 더 안정적으로 진행됨
# 주의: 실제 학습에는 원래 키 X가 아니라, 정규화된 입력값 X_norm을 사용
#       (X_mean, X_std는 나중에 '새 입력값 예측'에서도 똑같이 다시 씀)
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean:', X_mean)
print('입력값 표준편차 X_std:', X_std)
print('정규화된 입력값 X_norm:', X_norm)

입력값 평균 X_mean: 175.0
입력값 표준편차 X_std: 11.180339887498949
정규화된 입력값 X_norm: [-1.34164079 -0.4472136   0.4472136   1.34164079]


In [4]:
# 2-1. X_norm과 y를 PyTorch tensor로 변환하고 shape을 (n, 1)로 정리 (이전 파일과 동일)

# dtype=torch.float32 : 소수점 계산(미분)을 위해 실수(float) 형식으로 만듦
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# torch.nn.Linear(1, 1)에 넣으려면 각 데이터가 '입력 특성 1개'를 가진 형태,
# 즉 shape(n, 1)이어야 함. 그래서 reshape(-1, 1)로 모양을 바꿈
#   -1 : 행 개수는 알아서 (여기서는 4)
#    1 : 열 개수는 1 (입력 특성 1개)
X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

print('학습용 입력 tensor X_norm_tensor:', X_norm_tensor)
print('학습용 정답 tensor y_tensor', y_tensor)

# shape을 꼭 확인! 둘 다 (4, 1)이어야 함
print('X_norm_tensor shape:', X_norm_tensor.shape)
print('y_tensor shape:', y_tensor.shape)

학습용 입력 tensor X_norm_tensor: tensor([[-1.3416],
        [-0.4472],
        [ 0.4472],
        [ 1.3416]])
학습용 정답 tensor y_tensor tensor([[0.],
        [0.],
        [1.],
        [1.]])
X_norm_tensor shape: torch.Size([4, 1])
y_tensor shape: torch.Size([4, 1])


In [5]:
# 3. PerceptronModel 정의 (torch.nn.Module 상속)

# torch.nn.Module을 상속받아 우리만의 모델 class를 만듦
# 이전 파일에서 흩어져 있던 linear 생성과 H/z 계산을 이 안으로 모음
class PerceptronModel(torch.nn.Module):
    
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(1, 1)
    
    def forward(self, x):
        H = self.linear(x)
        
        z = torch.sigmoid(H)
        
        return z

In [14]:
torch.manual_seed(42)  # 랜덤 초기값 고정. 반드시 model 생성 '전'에 둠

model = PerceptronModel()

model.linear.weight
model.linear.bias

model

PerceptronModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)

In [15]:
with torch.no_grad():
    z_test = model(X_norm_tensor)
    
z_test

tensor([[0.4512],
        [0.6197],
        [0.7635],
        [0.8648]])

In [16]:
criterion = torch.nn.BCELoss()

criterion

BCELoss()

In [17]:
learning_rate = 0.1
epochs = 1000

In [19]:
list(model.parameters())

[Parameter containing:
 tensor([[0.7645]], requires_grad=True),
 Parameter containing:
 tensor([0.8300], requires_grad=True)]

In [20]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

optimizer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.1
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

In [22]:
for epoch in range(epochs):
    optimizer.zero_grad()  
    
    z = model(X_norm_tensor)
    
    mean_cost = criterion(z, y_tensor)
    
    mean_cost.backward()
    
    optimizer.step()
    
    if epoch % 100 == 0 or epoch == epochs-1:
        print(
            f'epoch={epoch}, '
            f'Cost={mean_cost.item():.6f}, '
            f'weight(a)={model.linear.weight.item():.6f}, '
            f'bias(b)={model.linear.bias.item():.6f}'
        )
        
    if epoch < 3:
        print(
            f'  (확인용) model.linear.weight.grad={model.linear.weight.grad.item():.6f}, '
            f'model.linear.bias.grad={model.linear.bias.grad.item():.6f}'
        )

epoch=0, Cost=0.495464, weight(a)=0.793780, bias(b)=0.812529
  (확인용) model.linear.weight.grad=-0.292415, model.linear.bias.grad=0.174793
  (확인용) model.linear.weight.grad=-0.286153, model.linear.bias.grad=0.169918
  (확인용) model.linear.weight.grad=-0.280072, model.linear.bias.grad=0.165171
epoch=100, Cost=0.178670, weight(a)=2.290082, bias(b)=0.173212
epoch=200, Cost=0.125357, weight(a)=3.002210, bias(b)=0.061586
epoch=300, Cost=0.099283, weight(a)=3.509002, bias(b)=0.026837
epoch=400, Cost=0.082901, weight(a)=3.912263, bias(b)=0.013229
epoch=500, Cost=0.071398, weight(a)=4.250606, bias(b)=0.007116
epoch=600, Cost=0.062789, weight(a)=4.543496, bias(b)=0.004091
epoch=700, Cost=0.056068, weight(a)=4.802371, bias(b)=0.002480
epoch=800, Cost=0.050660, weight(a)=5.034644, bias(b)=0.001570
epoch=900, Cost=0.046207, weight(a)=5.245449, bias(b)=0.001031
epoch=999, Cost=0.042507, weight(a)=5.436657, bias(b)=0.000701


In [23]:
# 학습된 weight, bias는 optimizer.step()에 의해 1000번 반복 업데이트된 값
# 입력 특성이 1개라 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냄
print('학습된 weight(a):', model.linear.weight.item())
print('학습된 bias(b):', model.linear.bias.item())

# tensor 원본 형태도 함께 확인해 둠. (shape과 requires_grad 표시를 볼 수 있음)
print('model.linear.weight:', model.linear.weight)
print('model.linear.bias:', model.linear.bias)

# 학습 후 grad도 한 번 확인해 봄. (마지막 epoch의 grad가 남아있음)
#   기존 grad_a -> model.linear.weight.grad
#   기존 grad_b -> model.linear.bias.grad
print('model.linear.weight.grad:', model.linear.weight.grad)
print('model.linear.bias.grad:', model.linear.bias.grad)

학습된 weight(a): 5.436656951904297
학습된 bias(b): 0.0007013267604634166
model.linear.weight: Parameter containing:
tensor([[5.4367]], requires_grad=True)
model.linear.bias: Parameter containing:
tensor([0.0007], requires_grad=True)
model.linear.weight.grad: tensor([[-0.0185]])
model.linear.bias.grad: tensor([2.6396e-05])


In [24]:
# 10. 새로운 입력값 예측

# 키가 185cm인 사람이 농구선수인지 예측
input_height = 185

# 새로운 입력값도 학습 데이터와 '같은 기준'으로 정규화해야 함
# 학습 때 사용한 X_mean, X_std를 그대로 다시 사용 (새로 계산하면 안 됨)
input_norm = (input_height - X_mean) / X_std

# 예측은 학습이 아니므로 a, b를 업데이트하지 않음
# 따라서 미분 계산 기록도 필요 없으므로 with torch.no_grad() 안에서 계산
with torch.no_grad():
    # torch.nn.Linear(1, 1)에 넣으려면 입력 shape을 (1, 1)로 맞춰야 함
    #   [[input_norm]] : 이중 대괄호로 감싸 (데이터 1개, 입력 특성 1개) = (1, 1) 형태로 만듦  
    input_norm_tensor = torch.tensor([[input_norm]], dtype=torch.float32)
    print('input_norm_tensor shape:', input_norm_tensor.shape)
    
    # z_new = model(input_norm_tensor)은 내부적으로
    #     H_new = self.linear(input_norm_tensor)   (선형 계산값 - 확률 아님)
    #     z_new = torch.sigmoid(H_new)             (예측 확률 - 0~1 사이)
    # 를 실행한 결과임. H_new가 직접 보이지 않을 뿐, 계산은 그대로 일어남.
    z_new = model(input_norm_tensor)
    
    # 0.5 이상이면 1(농구선수), 미만이면 0(농구선수 아님)
    # z_new는 shape (1, 1) tensor이므로 .item()으로 숫자 하나를 꺼냄
    pred = 1 if z_new.item() >= 0.5 else 0

print(f'키가 {input_height}cm인 사람이 농구선수일 확률(z): {z_new.item():.4f}')
if pred == 1:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')

input_norm_tensor shape: torch.Size([1, 1])
키가 185cm인 사람이 농구선수일 확률(z): 0.9923
판별 결과: 농구선수입니다.
